<a href="https://colab.research.google.com/github/lennartvoelz/fine_tune_hf/blob/main/funcgemma_finetune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Install Unsloth and other dependencies

In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.57.3
!pip install --no-deps trl==0.22.2

In [2]:
!pip install modelscope
os.environ['UNSLOTH_USE_MODELSCOPE'] = '1'

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 77.2 MB/s eta 0:00:00


### Unsloth

In [4]:
import torch
from google.colab import userdata
from datasets import load_dataset
from unsloth import FastLanguageModel
import json

hf_token = userdata.get('HF_TOKEN')
max_seq_length = 768
TARGET_DTYPE = torch.float16

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "google/functiongemma-270m-it",
    max_seq_length = max_seq_length,
    load_in_4bit = False,
    load_in_8bit = False,
    load_in_16bit = True,
    full_finetuning = False,
    torch_dtype=TARGET_DTYPE,
    token = hf_token,
)

2026-03-05 19:56:06,251 - modelscope - INFO - Got 11 files, start to download ...


Processing 11 items:   0%|          | 0.00/11.0 [00:00<?, ?it/s]

2026-03-05 20:03:54,144 - modelscope - INFO - Download model 'unsloth/functiongemma-270m-it' successfully.


`torch_dtype` is deprecated! Use `dtype` instead!


==((====))==  Unsloth 2026.3.3: Fast Gemma3 patching. Transformers: 4.57.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.


We now add LoRA adapters so we only need to update a small amount of parameters!

In [5]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 128, # Choose any number > 0! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 64,
    lora_dropout = 0.0, # Supports any, but = 0 is optimized
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 96,
    use_rslora = False,
    loftq_config = None,
)

Unsloth: Making `model.base_model.model.model` require gradients


In [6]:
from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = "/content/drive/MyDrive/data/"

Mounted at /content/drive


Formatting is very important in the `functiongemma` for tool-calling.

In [7]:
raw_data = []
with open(SAVE_DIR+"examples.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            raw_data.append(json.loads(line))

# ---------------------------------------------------------------------------
# Cactus C++ equivalent formatting (gemma_tools.h)
# ---------------------------------------------------------------------------

STANDARD_KEYS = {"description", "type", "properties", "required", "nullable"}


def _escape(s):
    return f"<escape>{s}<escape>"


def format_argument(value, escape_keys=True):
    if isinstance(value, str):
        return _escape(value)
    if isinstance(value, bool):
        return "true" if value else "false"
    if isinstance(value, dict):
        parts = []
        for k in sorted(value.keys()):
            key_str = _escape(k) if escape_keys else k
            parts.append(f"{key_str}:{format_argument(value[k], escape_keys)}")
        return "{" + ",".join(parts) + "}"
    if isinstance(value, (list, tuple)):
        return "[" + ",".join(format_argument(v, escape_keys) for v in value) + "]"
    if value is None:
        return "null"
    return str(value)


def _format_required_list(required):
    return "[" + ",".join(_escape(r) for r in required) + "]"


def format_parameters(properties, _required=None):
    parts = []
    for key in sorted(properties.keys()):
        if key in STANDARD_KEYS:
            continue

        prop = properties[key]
        if not isinstance(prop, dict):
            continue

        inner = []

        if "description" in prop:
            inner.append(f"description:{_escape(prop['description'])}")

        type_val = prop.get("type", "")

        if type_val.upper() == "STRING":
            if "enum" in prop:
                inner.append(f"enum:{format_argument(prop['enum'], escape_keys=True)}")

        elif type_val.upper() == "OBJECT":
            if "properties" in prop:
                inner.append(
                    "properties:{"
                    + format_parameters(prop["properties"], prop.get("required"))
                    + "}"
                )
            if "required" in prop:
                inner.append(f"required:{_format_required_list(prop['required'])}")

        elif type_val.upper() == "ARRAY":
            if "items" in prop and isinstance(prop["items"], dict):
                items_parts = []
                items = prop["items"]
                for item_key in sorted(items.keys()):
                    item_value = items[item_key]
                    if item_key == "properties":
                        items_parts.append(
                            "properties:{"
                            + format_parameters(item_value, items.get("required"))
                            + "}"
                        )
                    elif item_key == "required":
                        items_parts.append(f"required:{_format_required_list(item_value)}")
                    elif item_key == "type":
                        if isinstance(item_value, str):
                            items_parts.append(f"type:{_escape(item_value.upper())}")
                        elif isinstance(item_value, list):
                            items_parts.append(
                                "type:[" + ",".join(_escape(v.upper()) for v in item_value) + "]"
                            )
                    else:
                        items_parts.append(
                            f"{item_key}:{format_argument(item_value, escape_keys=True)}"
                        )
                inner.append("items:{" + ",".join(items_parts) + "}")

        if type_val:
            inner.append(f"type:{_escape(type_val.upper())}")

        parts.append(f"{key}:{{{','.join(inner)}}}")

    return ",".join(parts)


def format_function_declaration(name, description, params):
    result = f"declaration:{name}{{description:{_escape(description)}"
    if params:
        result += ",parameters:{"
        if "properties" in params:
            result += "properties:{" + format_parameters(params["properties"], params.get("required")) + "}"
        if "required" in params:
            result += f",required:{_format_required_list(params['required'])}"
        if "type" in params:
            result += f",type:{_escape(params['type'].upper())}"
        result += "}"
    result += "}"
    return result


def format_tools(tools):
    if not tools:
        return ""
    result = ""
    for tool in tools:
        t = tool["function"] if "function" in tool else tool
        result += "<start_function_declaration>"
        result += format_function_declaration(
            t.get("name", ""),
            t.get("description", ""),
            t.get("parameters", {}),
        )
        result += "<end_function_declaration>"
    return result


def format_function_call(call):
    name = call["name"]
    args = call.get("arguments", {})
    arg_parts = []
    for k in sorted(args.keys()):
        arg_parts.append(f"{k}:{format_argument(args[k], escape_keys=False)}")
    return (
        f"<start_function_call>call:{name}"
        + "{" + ",".join(arg_parts) + "}"
        + "<end_function_call>"
    )


def format_function_calls(calls):
    return "".join(format_function_call(c) for c in calls)


# ---------------------------------------------------------------------------
# Prompt construction (mirrors engine_tokenizer.cpp format_gemma_style)
# ---------------------------------------------------------------------------

def parse_tools_json(tools_json):
    tools = []
    if not tools_json or tools_json == "[]":
        return tools
    try:
        data = ast.literal_eval(tools_json)
    except Exception:
        try:
            data = json.loads(tools_json)
        except Exception:
            return tools
    for item in data:
        if isinstance(item, dict):
            tools.append(item)
    return tools


def format_tools_for_prompt(example):
    user_msg = next((msg["content"] for msg in example["messages"] if msg["role"] == "user"), "")
    expected_calls = example.get("expected_calls", [])
    tools_list = example.get("tools", [])

    # --- 1. BUILD THE SYSTEM PROMPT ---
    system_content = "You are a model that can do function calling with the following functions"
    tools_formatted = format_tools(tools_list) if tools_list else ""

    # --- 2. BUILD THE ASSISTANT CONTENT ---
    if expected_calls:
        assistant_content = format_function_calls(expected_calls)
    else:
        assistant_content = example.get("expected_response", "")

    # --- 3. CONSTRUCT THE FULL CHAT STRING ---
    chat_str = "<bos>"

    if tools_formatted:
        chat_str += f"<start_of_turn>developer\n{system_content}{tools_formatted}<end_of_turn>\n"
    else:
        chat_str += f"<start_of_turn>developer\n{system_content}<end_of_turn>\n"

    chat_str += f"<start_of_turn>user\n{user_msg}<end_of_turn>\n"

    # If the model is making a tool call, it MUST end with <start_function_response>
    if expected_calls:
        chat_str += f"<start_of_turn>model\n{assistant_content}<start_function_response>"
    else:
        chat_str += f"<start_of_turn>model\n{assistant_content}<end_of_turn>\n"

    return {"text": chat_str}

# ---------------------------------------------------------------------------
# Convert dataset
# ---------------------------------------------------------------------------

chat_data = []
for example in raw_data:
    formatted_example = format_tools_for_prompt(example)
    chat_data.append(formatted_example)

with open("chat_ready.jsonl", "w", encoding="utf-8") as f:
    for ex in chat_data:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")

print(f"Converted {len(chat_data)} examples -> chat_ready.jsonl")


Converted 100 examples -> chat_ready.jsonl


In [8]:
dataset = load_dataset("json", data_files="chat_ready.jsonl", split="train")

print(dataset[0])

Generating train split: 0 examples [00:00, ? examples/s]

{'text': '<bos><start_of_turn>developer\nYou are a model that can do function calling with the following functions<start_function_declaration>declaration:get_weather{description:<escape>Get current weather for a location<escape>,parameters:{properties:{location:{description:<escape>City name<escape>,type:<escape>STRING<escape>}},required:[<escape>location<escape>],type:<escape>OBJECT<escape>}}<end_function_declaration><end_of_turn>\n<start_of_turn>user\nWhat is the weather in San Francisco?<end_of_turn>\n<start_of_turn>model\n<start_function_call>call:get_weather{location:<escape>San Francisco<escape>}<end_function_call><start_function_response>'}


<a name="Train"></a>
### Train the model
Now let's train our model. We do 500 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`.

In [9]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    eval_dataset = None,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 4,
        num_train_epochs = 3,
        #max_steps = 15,
        learning_rate = 8e-5,
        logging_steps = 1,
        optim = "adamw_torch_fused",
        weight_decay = 0.007,
        lr_scheduler_type = "constant",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
        fp16=True,
        packing=False
    ),
)

Unsloth: Switching to float32 training since model cannot work with float16


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/100 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


We also use Unsloth's `train_on_completions` method to only train on the assistant outputs and ignore the loss on the user's inputs. This helps increase accuracy of finetunes!

In [10]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<start_of_turn>user\n",
    response_part = "<start_of_turn>model\n",
)

Map (num_proc=6):   0%|          | 0/100 [00:00<?, ? examples/s]

Filter (num_proc=6):   0%|          | 0/100 [00:00<?, ? examples/s]

Let's verify masking the instruction part is done! Let's print the 100th row again.

In [11]:
tokenizer.decode(trainer.train_dataset[-1]["input_ids"])

"<bos><bos><start_of_turn>developer\nYou are a model that can do function calling with the following functions<start_function_declaration>declaration:search_contacts{description:<escape>Search for a contact by name<escape>,parameters:{properties:{query:{description:<escape>Name to search for<escape>,type:<escape>STRING<escape>}},required:[<escape>query<escape>],type:<escape>OBJECT<escape>}}<end_function_declaration><start_function_declaration>declaration:send_message{description:<escape>Send a message to a contact<escape>,parameters:{properties:{message:{description:<escape>The message content to send<escape>,type:<escape>STRING<escape>},recipient:{description:<escape>Name of the person to send the message to<escape>,type:<escape>STRING<escape>}},required:[<escape>recipient<escape>,<escape>message<escape>],type:<escape>OBJECT<escape>}}<end_function_declaration><start_function_declaration>declaration:create_reminder{description:<escape>Create a reminder with a title and time<escape>,par

Now let's print the masked out example - you should see only the answer is present:

In [12]:
[tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[25]["labels"]]).replace(tokenizer.pad_token, "-")]

['---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------<start_function_call>call:set_alarm{hour:6,minute:45}<end_function_call><start_function_call>call:create_reminder{time:<escape>7:00 AM<escape>,title:<escape>take medicine<escape>}<end_function_call><start_function_response>']

Let's train the model! To resume a training run, set `trainer.train(resume_from_checkpoint = True)`

In [13]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 100 | Num Epochs = 3 | Total steps = 39
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 2 x 1) = 8
 "-____-"     Trainable parameters = 30,375,936 of 298,474,112 (10.18% trained)


Step,Training Loss
1,0.264500
2,0.009400
3,0.410000
4,0.087900
5,0.091900
6,0.099000
7,0.169700
8,0.028000
9,0.042800
10,0.121300


Unsloth: Will smartly offload gradients to save VRAM!


In [19]:
import json

raw_data = []
with open(SAVE_DIR+"examples.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            raw_data.append(json.loads(line))

def build_inference_prompt(user_msg, tools_list):
    """Build the generation prompt matching the exact FunctionGemma training template."""
    system_content = "You are a model that can do function calling with the following functions"
    tools_formatted = format_tools(tools_list) if tools_list else ""

    prompt = "<bos>"

    if tools_formatted:
        prompt += f"<start_of_turn>developer\n{system_content}{tools_formatted}<end_of_turn>\n"
    else:
        prompt += f"<start_of_turn>developer\n{system_content}<end_of_turn>\n"

    prompt += f"<start_of_turn>user\n{user_msg}<end_of_turn>\n"
    prompt += "<start_of_turn>model\n"

    return prompt

# ---------------------------------------------------------------------------
# Run inference
# ---------------------------------------------------------------------------

example = raw_data[20]
user_msg = next((msg["content"] for msg in example["messages"] if msg["role"] == "user"), "")
tools_list = example.get("tools", [])

text = build_inference_prompt(user_msg, tools_list)
inputs = tokenizer(text, return_tensors="pt").to("cuda")

eos_token_id = tokenizer.eos_token_id
start_resp_token_id = tokenizer.convert_tokens_to_ids("<start_function_response>")
stop_tokens = [eos_token_id, start_resp_token_id]

from transformers import TextStreamer
streamer = TextStreamer(tokenizer, skip_prompt=False)

_ = model.generate(
    **inputs,
    max_new_tokens=768,
    streamer=streamer,
    do_sample=False,
    temperature=0.0,
    eos_token_id=stop_tokens,
)

<bos><bos><start_of_turn>developer
You are a model that can do function calling with the following functions<start_function_declaration>declaration:get_weather{description:<escape>Get current weather for a location<escape>,parameters:{properties:{location:{description:<escape>City name<escape>,type:<escape>STRING<escape>}},required:[<escape>location<escape>],type:<escape>OBJECT<escape>}}<end_function_declaration><start_function_declaration>declaration:send_message{description:<escape>Send a message to a contact<escape>,parameters:{properties:{message:{description:<escape>The message content to send<escape>,type:<escape>STRING<escape>},recipient:{description:<escape>Name of the person to send the message to<escape>,type:<escape>STRING<escape>}},required:[<escape>recipient<escape>,<escape>message<escape>],type:<escape>OBJECT<escape>}}<end_function_declaration><start_function_declaration>declaration:set_alarm{description:<escape>Set an alarm for a given time<escape>,parameters:{properties

In [15]:
model.save_pretrained(SAVE_DIR+"lora_dir")
tokenizer.save_pretrained(SAVE_DIR+"lora_dir")

('/content/drive/MyDrive/data/lora_dir/tokenizer_config.json',
 '/content/drive/MyDrive/data/lora_dir/special_tokens_map.json',
 '/content/drive/MyDrive/data/lora_dir/chat_template.jinja',
 '/content/drive/MyDrive/data/lora_dir/tokenizer.model',
 '/content/drive/MyDrive/data/lora_dir/added_tokens.json',
 '/content/drive/MyDrive/data/lora_dir/tokenizer.json')

In [17]:
from peft import PeftModel, PeftConfig

base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name = "google/functiongemma-270m-it",
    max_seq_length = max_seq_length,
    load_in_4bit = False,
    load_in_8bit = False,
    load_in_16bit = True,
    full_finetuning = False,
    torch_dtype=TARGET_DTYPE,
    token = hf_token,
)
peft_model = PeftModel.from_pretrained(base_model, SAVE_DIR+"lora_dir")
merged_model = peft_model.merge_and_unload()


==((====))==  Unsloth 2026.3.3: Fast Gemma3 patching. Transformers: 4.57.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.


In [18]:
repo_id = "lvoelz/funcgemma_v2"
merged_model.push_to_hub(repo_id, token=hf_token)
base_tokenizer.push_to_hub(repo_id, token=hf_token)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...29bh4zu/model.safetensors:   1%|1         | 7.89MB /  536MB            

Saved model to https://huggingface.co/lvoelz/funcgemma_v2


README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...pzex38pa5/tokenizer.model: 100%|##########| 4.69MB / 4.69MB            

  ...mpzex38pa5/tokenizer.json:  24%|##3       | 7.97MB / 33.4MB            